## **16. 특징 선택**

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
from matplotlib import pyplot as plt
import seaborn as sns

In [ ]:
# 타임스탬프 파싱 함수

def parser(x):
    return datetime.strptime(x, '%Y-%m-%d %H:%M:%S')


In [ ]:
# 데이터셋 로딩

path = './data/'
file_name = 'AirQualityUCI_refined.csv'

df = pd.read_csv(
    path+file_name,
    index_col=[0],
    parse_dates=[0],
    date_parser=parser,
    dtype='float32'
)

df.head()

In [ ]:
# # 시각화 설정 옵션 (qt5 설치된 환경에서만 실행)
# %matplotlib qt5
# %config InlineBackend.figure_format = 'svg'

# plt.rcParams['figure.figsize'] = [12, 5]
# plt.rcParams['font.size'] = 13
# plt.ion()

In [ ]:
# [+] 전체 변수에 대한 결측치 처리 (선형 보간)

df.interpolate(inplace=True)
df.info()

In [ ]:
# 상관행렬 시각화
sns.pairplot(
    df, 
    kind='reg', 
    diag_kind='kde', 
    plot_kws={'scatter_kws': {'alpha': 0.1}}
)

In [ ]:
# [+] 상관계수 측정
df.corr()

**모델 학습 및 특징 중요도 출력**

특징 집합을 훈련 데이터로 학습하여 일산화탄소를 예측하는 회귀 모델
- 레이블: 일산화탄소(`CO(GT)`)
- 특징 집합: `CO(GT)`를 제외한 나머지 변수

In [ ]:
# [+] 학습 데이터셋 준비
X = df.iloc[:, 1:]  # 훈련 데이터
y = df.iloc[:, 0]  # 레이블 데이터
print(X.shape, y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

# [+] 훈련 / 테스트 분할
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.ensemble import RandomForestRegressor  # 랜덤 포레스트 회귀 모형

model = RandomForestRegressor()  # [+] 모델 객체 생성
model.fit(X_train, y_train)  # [+] 모델 학습

In [ ]:
# [+] 예측 수행
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import mean_absolute_error

# [+] MAE 계산
mae = mean_absolute_error(y_test, y_pred)
print(f"MAE (Mean Absolute Error): {mae:.4f}")

In [ ]:
# [+] 특징 중요도 출력
model.feature_importances_

In [ ]:
# 특징 중요도 계산
feat_importances = pd.Series(model.feature_importances_, index=X.columns)
feat_importances

In [ ]:
# 특징 중요도 시각화
feat_importances.sort_values(ascending=True).plot(kind='barh')
plt.show()